# Test `oncophenomap`

This notebook fits the initial genotype–phenotype tree on the synthetic CRC cohort
deposited on Zenodo (`synthetic_cohort_data.zip`, https://doi.org/10.5281/zenodo.23055294).
Extract the archive and set `SYNTHCRC_SYNTHETIC_DIR` to the extracted `synthetic_cohort_data`
folder before running. The cohort contains no patient data; sample identifiers are synthetic
(`syn_######`).

In [ ]:
from pathlib import Path
import sys
import pandas as pd

repo = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo))

import importlib
import oncophenomap as op
import oncophenomap.plotting as op_plotting

importlib.reload(op)
importlib.reload(op_plotting)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# The extracted Zenodo synthetic cohort (no patient data).
import os
SYNTHETIC_DIR = Path(os.environ['SYNTHCRC_SYNTHETIC_DIR'])
DNA_FILE = SYNTHETIC_DIR / 'mutations.parquet'
FACTOR_SCORES_FILE = SYNTHETIC_DIR / 'factor_scores.parquet'
ARM_FILE = SYNTHETIC_DIR / 'arm_fractions.parquet'
HE_FILE = SYNTHETIC_DIR / 'he_features.parquet'
FOCAL_CNV_FILE = SYNTHETIC_DIR / 'focal_cnv.parquet'
CLINICAL_FILE = SYNTHETIC_DIR / 'clinical.parquet'
TAXONOMY_FILE = SYNTHETIC_DIR / 'taxonomy.parquet'
MIN_MUTATION_PREVALENCE = 0.01
MIN_CNV_TREE_PREVALENCE = 0.01
# The synthetic cohort does not include a gene-coordinate reference, so CNV-locus
# collapsing (which requires one -- see oncophenomap.cnv.collapse_selected_cnv_loci) is
# left disabled below (`collapse_cnv_loci=False`).

for path in [DNA_FILE, FACTOR_SCORES_FILE, ARM_FILE, HE_FILE, FOCAL_CNV_FILE, CLINICAL_FILE, TAXONOMY_FILE]:
    print(f'{path}: {path.exists()}')

In [ ]:
mutations = pd.read_parquet(DNA_FILE)
clinical = pd.read_parquet(CLINICAL_FILE)
taxonomy = pd.read_parquet(TAXONOMY_FILE)
mutations = mutations.loc[:, mutations.mean() >= MIN_MUTATION_PREVALENCE].copy()
arms = pd.read_parquet(ARM_FILE)
he = pd.read_parquet(HE_FILE)

phenotype = pd.read_parquet(FACTOR_SCORES_FILE)
if 'dense_0' in phenotype.columns:
    phenotype = phenotype.drop(columns='dense_0')

focal_cnv = pd.read_parquet(FOCAL_CNV_FILE)
focal_cnv = op.focal_cnv_features(focal_cnv, MIN_CNV_TREE_PREVALENCE)

print('mutations:', mutations.shape)
print('phenotype:', phenotype.shape)
print('arms:', arms.shape)
print('H&E:', he.shape)
print('focal CNV:', focal_cnv.shape)
print('clinical:', clinical.shape)
print('taxonomy:', taxonomy.shape)

In [ ]:
# Start with point mutations. Focal CNV can be concatenated as an additional
# genotype block after applying any desired prevalence filtering.
# H&E does not define cohort membership: missing H&E samples are retained.
common = phenotype.index.intersection(mutations.index).intersection(arms.index)
Y = phenotype.loc[common]
X = mutations.loc[common]
A = arms.loc[common]
H = he.reindex(common)

metadata_strips = pd.DataFrame(index=common)
if 'heFibroblasts' in H.columns:
    metadata_strips['heFibroblasts'] = pd.to_numeric(H['heFibroblasts'], errors='coerce')
metadata_fields = [field for field in ['heFibroblasts'] if field in metadata_strips.columns]

# Optional: include focal CNV features in the genotype matrix.
cnv = focal_cnv.reindex(common).fillna(0)
X_with_cnv = pd.concat([X, cnv], axis=1)
X_with_cnv = X_with_cnv.loc[:, ~X_with_cnv.columns.duplicated()]
print('aligned samples:', len(common))
print('metadata strips:', metadata_fields)

In [ ]:
model = op.GenotypePhenotypeTree(
    max_leaf_nodes=100,
    min_samples_leaf=25,
    random_state=42,
).fit_with_context(
    phenotype=Y,
    genotype=X_with_cnv,
    arm_fractions=A,
    he_features=H,
    residualize=['arms', 'he'],
    collapse_cnv_loci=False,
    verbose=True,
)

In [ ]:
#model.estimate_heldout_group_variance()
model.summary()

In [ ]:
print(f'Variance explained by genotype tree: {model.variance_explained_:.3%}')
print(f'Number of groups: {model.labels_.nunique()}')
model.feature_importances_.sort_values(ascending=False).head(15)

In [ ]:
# This example uses factor-mode (row_mode="factor"). For program-mode, build
# `factor_groups` from the `program` column of the cohort's factor_reference.parquet
# (see README "Program-mode heatmap").
core = op.build_group_heatmap_data(
    phenotype=Y,
    group_labels=model.labels_,
    group_paths=model.leaf_rules_,
    group_order="tree",
    row_mode="factor",
    tree_model=model,
    strip_features=X_with_cnv,
    arm_features=A,
    append_top_prevalent_arm_strips_n=0,
    metadata_features=metadata_strips,
    metadata_strip_fields=metadata_fields,
    metadata_strip_modes={
        'heFibroblasts': 'quantile',
    },
    metadata_strip_labels={
        'heFibroblasts': 'Fibroblasts (H&E)',
    },
    top_n=25,
    include_group_size_strip=True,
)

In [ ]:
fig, ax = op.plot_group_heatmap(
    core,
    title="Group heatmap",
    cmap="RdYlBu_r",
    strips_ratio=0.012,
    show_strips=True,
    show_annotation_block_separators=True,
    show_annotation_block_borders=True,
    show_tree_branch_mutation_markers=True,
    mutation_prevalence_saturation=0.5,
    figsize=(15, 9),
)